[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/11.%20The%20Dynamic%20Berth%20Allocation%20Problem/P11-Tier-1_executed.ipynb) [![Open In SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 11. The Dynamic Berth Allocation Problem
## Tier 1 — Mathematical Formulation

### Goal and scope
This notebook formulates the Dynamic Berth Allocation Problem (DBAP) as a time-indexed integer programming model. Given vessel arrival times, vessel lengths, berth capacities, and vessel–berth processing times, the model assigns each vessel to one feasible berth and integer start time. It minimizes a weighted sum of waiting and processing time.

The inputs are the vessel and berth data defined below, together with a discrete planning horizon and objective weights. The outputs are a feasible schedule, its waiting and processing times, the objective value, a schedule visualization, and a sensitivity analysis of the objective weights. PuLP with the CBC solver is used to solve the integer program.

### Model assumptions
- Arrival times, processing times, and start times are measured in integer hours and are known in advance.
- A vessel cannot start service before its arrival.
- A vessel occupies one berth continuously from its start time through, but not including, its completion time.
- Each berth can serve at most one vessel at any instant. Vessel length must not exceed the capacity of its assigned berth.
- Processing time depends on the vessel–berth combination.
- The planning horizon is 24 hours in the main example. Assignments that would finish after the horizon are excluded.
- The data include a desired departure time, but this model does not impose a departure deadline or penalize departures after that time. It also does not model spatial positioning along a quay, uncertainty, tides, or changes in data over time.

### Decision variable and objective
The binary decision variable records whether vessel \(i\) starts at berth \(j\) at integer time \(t\):

$$
x_{ijt} =
\begin{cases}
1, & \text{if vessel } i \text{ starts at berth } j \text{ at time } t,\\
0, & \text{otherwise.}
\end{cases}
$$

For vessel \(i\), let \(a_i\) be its arrival time, and let \(p_{ij}\) be its processing time at berth \(j\). The start time \(s_i\) and completion time \(c_i\) are determined by the selected assignment. The model minimizes weighted waiting time plus weighted processing time:

$$
\min \quad \sum_i \left[\alpha(s_i-a_i)+\beta(c_i-s_i)\right].
$$

Here \(s_i-a_i\) is vessel \(i\)'s waiting time, \(c_i-s_i\) is its processing time, and \(\alpha\) and \(\beta\) are their respective nonnegative weights. In the main example, both weights are 1, so the objective is total waiting time plus total processing time. This is also called total service time in the output; it is not the sum of completion times.

### Constraints
Each vessel must receive exactly one feasible berth and start time. Start and completion times are linked to the selected assignment by:

$$
\sum_{j,t} x_{ijt}=1 \quad \forall i,
$$

$$
s_i=\sum_{j,t} t x_{ijt} \quad \forall i,
\qquad
c_i=\sum_{j,t}(t+p_{ij})x_{ijt} \quad \forall i.
$$

Only assignments satisfying \(t\geq a_i\), \(t+p_{ij}\leq H\), and vessel length no greater than berth capacity are created, where \(H\) is the planning horizon. These conditions enforce arrival, horizon, and berth-capacity feasibility.

To express berth capacity in the discrete-time model, consider each berth \(j\) and each integer time slot \(q\in\{0,\ldots,H-1\}\). A vessel starting at \(t\) with processing time \(p_{ij}\) occupies slot \(q\) exactly when \(t\leq q<t+p_{ij}\). Therefore:

$$
\sum_{i,t:\,t\leq q<t+p_{ij}} x_{ijt} \leq 1
\quad \forall j,\ q\in\{0,\ldots,H-1\}.
$$

This time-slot capacity constraint prevents two vessels from occupying the same berth at the same time. The code implements it by summing the assignment variables whose service intervals cover each slot.

### Solution sequence and interpretation
The notebook defines the input data, constructs the binary assignment variables, adds the objective and constraints, solves the integer program, and extracts the resulting schedule. A mixed-integer solver may return one of several schedules if multiple schedules attain the same optimal objective value; an optimal schedule is not necessarily unique. The sensitivity analysis changes the weights to examine how the trade-off in the objective affects the schedule and its reported time totals.

In [ ]:
# Load the data and tools used to build, solve, and interpret the integer program.
import matplotlib.pyplot as plt
import pandas as pd
import pulp
from typing import Dict, List


In [ ]:
class Vessel:
    """Store a vessel's data used by the berth-allocation model."""

    def __init__(
        self,
        id: int,
        arrival_time: int,
        desired_departure: int,
        length: int,
        teu_capacity: int,
        processing_times: Dict[int, int],
    ):
        self.id = id
        self.arrival_time = arrival_time
        self.desired_departure = desired_departure
        self.length = length
        self.teu_capacity = teu_capacity
        self.processing_times = processing_times

    def __repr__(self):
        return (
            f"Vessel{self.id}(arrival={self.arrival_time}, "
            f"length={self.length}m, TEU={self.teu_capacity})"
        )


class Berth:
    """Store a berth's length capacity."""

    def __init__(self, id: int, capacity: int):
        self.id = id
        self.capacity = capacity

    def __repr__(self):
        return f"Berth{self.id}(capacity={self.capacity}m)"


# Processing times are specified separately for each vessel–berth pair.
vessels = [
    Vessel(1, 0, 10, 200, 5000, {1: 4, 2: 5}),
    Vessel(2, 2, 12, 250, 8000, {1: 3, 2: 4}),
    Vessel(3, 4, 15, 300, 12000, {1: 6, 2: 5}),
]

berths = [
    Berth(1, 350),
    Berth(2, 400),
]

print("Problem setup:")
print("Vessels:")
for vessel in vessels:
    print(f"  {vessel}")

print("\nBerths:")
for berth in berths:
    print(f"  {berth}")

In [ ]:
class DBAPOptimalSolver:
    """Build and solve the time-indexed integer program for the DBAP."""

    def __init__(
        self,
        vessels: List[Vessel],
        berths: List[Berth],
        time_horizon: int = 24,
        alpha: float = 1.0,
        beta: float = 1.0,
    ):
        self.vessels = vessels
        self.berths = berths
        self.time_horizon = time_horizon
        self.alpha = alpha
        self.beta = beta
        self.x = {}
        self.start_vars = {}
        self.completion_vars = {}
        self.model = None
        self.solution = None

    def create_model(self):
        """Create feasible binary assignments and vessel time variables."""
        if self.time_horizon <= 0:
            raise ValueError("The planning horizon must be positive.")
        if self.alpha < 0 or self.beta < 0:
            raise ValueError("Objective weights must be nonnegative.")

        self.model = pulp.LpProblem("DBAP", pulp.LpMinimize)

        # An assignment variable exists only if arrival, length, and horizon allow it.
        for vessel in self.vessels:
            for berth in self.berths:
                if vessel.length > berth.capacity:
                    continue
                if berth.id not in vessel.processing_times:
                    raise ValueError(
                        f"Missing processing time for vessel {vessel.id} "
                        f"at berth {berth.id}."
                    )

                processing_time = vessel.processing_times[berth.id]
                if processing_time <= 0:
                    raise ValueError("Processing times must be positive.")

                for start_time in range(self.time_horizon):
                    if (
                        start_time >= vessel.arrival_time
                        and start_time + processing_time <= self.time_horizon
                    ):
                        key = (vessel.id, berth.id, start_time)
                        self.x[key] = pulp.LpVariable(
                            f"x_{vessel.id}_{berth.id}_{start_time}",
                            cat="Binary",
                        )

        # These variables make the objective and schedule output explicit.
        for vessel in self.vessels:
            self.start_vars[vessel.id] = pulp.LpVariable(
                f"s_{vessel.id}", lowBound=0, upBound=self.time_horizon, cat="Integer"
            )
            self.completion_vars[vessel.id] = pulp.LpVariable(
                f"c_{vessel.id}", lowBound=0, upBound=self.time_horizon, cat="Integer"
            )

        return True

    def add_objective_function(self):
        """Minimize weighted waiting time plus weighted processing time."""
        if self.model is None:
            return False

        objective = pulp.lpSum(
            self.alpha * (self.start_vars[vessel.id] - vessel.arrival_time)
            + self.beta
            * (self.completion_vars[vessel.id] - self.start_vars[vessel.id])
            for vessel in self.vessels
        )
        self.model.setObjective(objective)
        return True

    def add_constraints(self):
        """Add assignment, time-definition, and berth-capacity constraints."""
        if self.model is None:
            return False

        # Every vessel must select exactly one feasible berth and start time.
        for vessel in self.vessels:
            assignment_vars = [
                self.x[key]
                for key in self.x
                if key[0] == vessel.id
            ]
            self.model += (
                pulp.lpSum(assignment_vars) == 1,
                f"assign_vessel_{vessel.id}",
            )

            # The selected assignment determines both reported vessel times.
            start_expr = pulp.lpSum(
                start_time * self.x[(vessel.id, berth.id, start_time)]
                for berth in self.berths
                for start_time in range(self.time_horizon)
                if (vessel.id, berth.id, start_time) in self.x
            )
            completion_expr = pulp.lpSum(
                (
                    start_time + vessel.processing_times[berth.id]
                )
                * self.x[(vessel.id, berth.id, start_time)]
                for berth in self.berths
                for start_time in range(self.time_horizon)
                if (vessel.id, berth.id, start_time) in self.x
            )
            self.model += (
                self.start_vars[vessel.id] == start_expr,
                f"start_time_{vessel.id}",
            )
            self.model += (
                self.completion_vars[vessel.id] == completion_expr,
                f"completion_time_{vessel.id}",
            )

        # A berth's capacity is one vessel in every integer time slot.
        # A service interval [start, start + processing) covers slot q when
        # start <= q < start + processing.
        for berth in self.berths:
            for time_slot in range(self.time_horizon):
                occupying_vars = []
                for vessel in self.vessels:
                    processing_time = vessel.processing_times.get(berth.id)
                    if processing_time is None:
                        continue
                    for start_time in range(self.time_horizon):
                        key = (vessel.id, berth.id, start_time)
                        if (
                            key in self.x
                            and start_time <= time_slot
                            and time_slot < start_time + processing_time
                        ):
                            occupying_vars.append(self.x[key])

                self.model += (
                    pulp.lpSum(occupying_vars) <= 1,
                    f"berth_capacity_{berth.id}_{time_slot}",
                )

        return True

    def solve(self, time_limit: int = 60):
        """Build the model, solve it with CBC, and return an optimal schedule."""
        if not self.create_model():
            return None
        if not self.add_objective_function():
            return None
        if not self.add_constraints():
            return None

        try:
            self.model.solve(
                pulp.PULP_CBC_CMD(timeLimit=time_limit, msg=0)
            )
        except Exception as error:
            print(f"Solver error: {error}")
            return None

        status = pulp.LpStatus[self.model.status]
        if status == "Optimal":
            self.extract_solution()
            return self.solution

        print(f"Model status: {status}")
        return None

    def extract_solution(self):
        """Convert selected assignment variables into a readable schedule."""
        self.solution = []

        for vessel in self.vessels:
            selected_assignments = [
                (berth, start_time)
                for berth in self.berths
                for start_time in range(self.time_horizon)
                if (vessel.id, berth.id, start_time) in self.x
                and pulp.value(self.x[(vessel.id, berth.id, start_time)]) > 0.5
            ]
            if len(selected_assignments) != 1:
                raise RuntimeError(
                    f"Expected one selected assignment for vessel {vessel.id}."
                )

            berth, start_time = selected_assignments[0]
            processing_time = vessel.processing_times[berth.id]
            completion_time = start_time + processing_time
            waiting_time = start_time - vessel.arrival_time

            self.solution.append(
                {
                    "vessel_id": vessel.id,
                    "berth_id": berth.id,
                    "start_time": start_time,
                    "completion_time": completion_time,
                    "processing_time": processing_time,
                    "waiting_time": waiting_time,
                    "service_time": waiting_time + processing_time,
                }
            )

        self.solution.sort(key=lambda row: (row["start_time"], row["vessel_id"]))


print("Integer programming model class is ready.")

In [ ]:
# Solve the concrete example with the integer programming model.
time_horizon = 24
solver = DBAPOptimalSolver(vessels, berths, time_horizon=time_horizon)
solution = solver.solve()

if solution is not None:
    df_solution = pd.DataFrame(solution).sort_values(
        ["start_time", "vessel_id"]
    )

    print("Optimal schedule:")
    print("=" * 50)
    for _, row in df_solution.iterrows():
        print(
            f"Vessel {int(row['vessel_id'])}: Berth {int(row['berth_id'])}, "
            f"Start {int(row['start_time'])}, "
            f"Complete {int(row['completion_time'])}, "
            f"Wait {int(row['waiting_time'])} h, "
            f"Process {int(row['processing_time'])} h"
        )

    total_service_time = df_solution["service_time"].sum()
    total_waiting_time = df_solution["waiting_time"].sum()
    total_processing_time = df_solution["processing_time"].sum()
    objective_value = pulp.value(solver.model.objective)
    calculated_objective = (
        solver.alpha * total_waiting_time
        + solver.beta * total_processing_time
    )

    assert abs(objective_value - calculated_objective) <= 1e-6
    print("\nSchedule summary:")
    print(f"Total waiting time: {total_waiting_time} hours")
    print(f"Total processing time: {total_processing_time} hours")
    print(f"Total service time: {total_service_time} hours")
    print(f"Weighted objective value: {objective_value}")
else:
    print(
        "No optimal schedule is available. Check the solver status, "
        "solver installation, and model feasibility."
    )

In [ ]:
def visualize_schedule(solution, vessels, berths, time_horizon=24):
    """Plot the schedule and report vessel and berth time summaries."""
    if not solution:
        print("There is no schedule to visualize.")
        return

    df_solution = pd.DataFrame(solution)
    fig, ax = plt.subplots(figsize=(12, 6))
    color_map = plt.get_cmap("tab10")

    for index, vessel in enumerate(vessels):
        vessel_data = df_solution[
            df_solution["vessel_id"] == vessel.id
        ]
        if vessel_data.empty:
            continue

        row = vessel_data.iloc[0]
        berth_id = int(row["berth_id"])
        start_time = int(row["start_time"])
        processing_time = int(row["processing_time"])
        color = color_map(index % color_map.N)

        ax.barh(
            berth_id,
            processing_time,
            left=start_time,
            color=color,
            alpha=0.8,
            label=f"Vessel {vessel.id} ({vessel.teu_capacity} TEU)",
        )
        ax.text(
            start_time + processing_time / 2,
            berth_id,
            f"V{vessel.id}",
            ha="center",
            va="center",
            fontweight="bold",
            color="white",
        )

        ax.axvline(
            x=vessel.arrival_time,
            color="red",
            linestyle="--",
            alpha=0.35,
            label="Vessel arrival" if index == 0 else None,
        )
        ax.text(
            vessel.arrival_time,
            len(berths) + 0.35,
            f"V{vessel.id} arrival",
            rotation=90,
            ha="right",
            va="bottom",
            fontsize=8,
        )

    ax.set_xlabel("Time (hours)")
    ax.set_ylabel("Berth")
    ax.set_title("Dynamic Berth Allocation Schedule")
    ax.set_yticks([berth.id for berth in berths])
    ax.set_yticklabels([f"Berth {berth.id}" for berth in berths])
    ax.set_xlim(0, time_horizon)
    ax.grid(True, axis="x", alpha=0.3)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()
    plt.show()

    print("\nDetailed schedule analysis:")
    summary_data = []
    for vessel in vessels:
        vessel_data = df_solution[
            df_solution["vessel_id"] == vessel.id
        ]
        if vessel_data.empty:
            continue

        row = vessel_data.iloc[0]
        summary_data.append(
            [
                vessel.id,
                vessel.teu_capacity,
                vessel.arrival_time,
                int(row["berth_id"]),
                int(row["start_time"]),
                int(row["completion_time"]),
                int(row["waiting_time"]),
                int(row["processing_time"]),
                int(row["service_time"]),
            ]
        )

    summary_df = pd.DataFrame(
        summary_data,
        columns=[
            "Vessel ID",
            "TEU Capacity",
            "Arrival Time (hours)",
            "Berth ID",
            "Start Time (hours)",
            "Completion Time (hours)",
            "Waiting Time (hours)",
            "Processing Time (hours)",
            "Service Time (hours)",
        ],
    )
    print(summary_df.to_string(index=False))

    print("\nBerth utilization during the planning horizon:")
    for berth in berths:
        berth_data = df_solution[df_solution["berth_id"] == berth.id]
        total_processing = berth_data["processing_time"].sum()
        utilization = total_processing / time_horizon * 100
        print(
            f"Berth {berth.id}: {total_processing} processing hours "
            f"({utilization:.1f}% of {time_horizon} hours)"
        )


visualize_schedule(solution, vessels, berths, time_horizon=time_horizon)

In [ ]:
def sensitivity_analysis():
    """Compare schedule totals under different objective weights."""
    print("Sensitivity analysis: objective-weight scenarios")
    print("=" * 55)

    scenarios = [
        {"name": "Base case", "alpha": 1.0, "beta": 1.0},
        {"name": "High waiting penalty", "alpha": 2.0, "beta": 1.0},
        {"name": "High processing penalty", "alpha": 1.0, "beta": 2.0},
        {"name": "Balanced penalties", "alpha": 1.5, "beta": 1.5},
    ]

    results = []
    for scenario in scenarios:
        scenario_solver = DBAPOptimalSolver(
            vessels,
            berths,
            time_horizon=time_horizon,
            alpha=scenario["alpha"],
            beta=scenario["beta"],
        )
        scenario_solution = scenario_solver.solve()

        if scenario_solution is None:
            results.append(
                [
                    scenario["name"],
                    scenario["alpha"],
                    scenario["beta"],
                    None,
                    None,
                    None,
                    None,
                ]
            )
            continue

        scenario_df = pd.DataFrame(scenario_solution)
        total_waiting = scenario_df["waiting_time"].sum()
        total_processing = scenario_df["processing_time"].sum()
        total_service = scenario_df["service_time"].sum()
        objective_value = pulp.value(scenario_solver.model.objective)
        results.append(
            [
                scenario["name"],
                scenario["alpha"],
                scenario["beta"],
                total_waiting,
                total_processing,
                total_service,
                objective_value,
            ]
        )

    results_df = pd.DataFrame(
        results,
        columns=[
            "Scenario",
            "Alpha (waiting)",
            "Beta (processing)",
            "Total waiting time (hours)",
            "Total processing time (hours)",
            "Total service time (hours)",
            "Weighted objective value",
        ],
    )
    print(results_df.to_string(index=False))

    feasible_results = [row for row in results if row[3] is not None]
    if feasible_results:
        scenario_names = [row[0] for row in feasible_results]
        waiting_times = [row[3] for row in feasible_results]
        processing_times = [row[4] for row in feasible_results]

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        ax1.bar(scenario_names, waiting_times, color="skyblue", alpha=0.8)
        ax1.set_title("Total Waiting Time by Scenario")
        ax1.set_ylabel("Hours")
        ax1.set_xlabel("Objective-weight scenario")
        ax1.tick_params(axis="x", rotation=35)

        ax2.bar(scenario_names, processing_times, color="lightcoral", alpha=0.8)
        ax2.set_title("Total Processing Time by Scenario")
        ax2.set_ylabel("Hours")
        ax2.set_xlabel("Objective-weight scenario")
        ax2.tick_params(axis="x", rotation=35)

        fig.tight_layout()
        plt.show()


sensitivity_analysis()

## Interpreting the formulation

### What the solution establishes
An optimal solver status means that the integer programming solver has found a schedule satisfying the model constraints and has certified that no feasible schedule has a smaller weighted objective value. The guarantee concerns this mathematical model and its supplied data; it does not establish that the model includes every operational detail of a real port. If more than one schedule has the same minimum objective, the solver may return any one of them.

For each vessel, the schedule reports its assigned berth, start and completion times, waiting time, processing time, and service time. Waiting time is start time minus arrival time. Processing time is completion time minus start time. Service time is their sum. The objective value applies the selected weights to waiting and processing totals, so it equals total service time only when both weights are 1.

### Computational size
With \(n\) vessels, \(m\) berths, and an integer horizon of \(H\) time slots, there are at most \(nmH\) binary assignment variables; feasibility filtering can reduce this number. The model also has one berth-capacity constraint for each berth and time slot, giving \(mH\) such constraints, in addition to assignment and time-definition constraints. Integer programming can require exponential computation in the worst case, although practical solve time depends on the instance and solver.

### Strengths and limitations
The formulation makes the objective and constraints explicit, provides an optimality certificate when the solver reports an optimal status, and can serve as a benchmark for the stated model. Its exactness depends on accurate input data and on the assumptions in the scope section. The time-indexed representation grows with the planning horizon, and the deterministic model treats arrivals and processing times as known with certainty. It does not include operational features such as uncertain arrivals, quay-side spatial positioning, or departure-time penalties.

### When this formulation is useful
This exact model is useful for small instances, for decisions where model-optimal schedules are needed, and for teaching how binary variables, objective functions, and capacity constraints represent a scheduling problem. For larger or more detailed applications, the model's scope and computational requirements should be evaluated against the decision-making context.